# 🛰️ PRAHARI — Real Sentinel-1 SAR Vessel Detection

Runs free on Google Colab. It pulls a **real Sentinel-1 radar image** of the Bay of Bengal from the Copernicus Data Space, detects ships with a CFAR-style bright-target detector (the classic SAR method — vessels are bright on dark sea), and prints the detections as **JSON to paste back into PRAHARI**.

**Before running**, create a Sentinel Hub OAuth client in the Copernicus dashboard and have your `client_id` + `client_secret` ready (see the chat instructions).

Run each cell top to bottom with **Shift+Enter**.

In [ ]:
# 1) Install dependencies
!pip -q install rasterio requests numpy scipy

In [ ]:
# 2) Enter your Copernicus (CDSE) Sentinel Hub OAuth credentials.
# These stay in THIS Colab session only — they are never sent anywhere except
# Copernicus's own login server.
import getpass
CLIENT_ID = getpass.getpass('CDSE Sentinel Hub client_id: ').strip()
CLIENT_SECRET = getpass.getpass('CDSE Sentinel Hub client_secret: ').strip()

In [ ]:
# 3) Get an access token from Copernicus
import requests
resp = requests.post(
    'https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token',
    data={'grant_type': 'client_credentials', 'client_id': CLIENT_ID, 'client_secret': CLIENT_SECRET},
)
tok = resp.json()
assert 'access_token' in tok, f'Login failed: {tok}'
TOKEN = tok['access_token']
print('Token OK ✓')

In [ ]:
# 4) Request a real Sentinel-1 VV radar image over a Bay of Bengal AOI
# (Chattogram approach / shipping lane — open water with traffic).
BBOX = [90.2, 20.2, 90.9, 20.9]  # [lon_min, lat_min, lon_max, lat_max]
WIDTH = HEIGHT = 1500            # ~50 m/pixel

evalscript = '''//VERSION=3
function setup(){ return { input:["VV","dataMask"], output:{ bands:2, sampleType:"FLOAT32" } }; }
function evaluatePixel(s){ return [s.VV, s.dataMask]; }'''

body = {
  "input": {
    "bounds": {"bbox": BBOX, "properties": {"crs": "http://www.opengis.net/def/crs/EPSG/0/4326"}},
    "data": [{"type": "sentinel-1-grd",
               "dataFilter": {"timeRange": {"from": "2024-01-01T00:00:00Z", "to": "2024-12-31T23:59:59Z"},
                                "mosaickingOrder": "mostRecent"},
               "processing": {"backCoeff": "SIGMA0_ELLIPSOID", "orthorectify": True}}]
  },
  "output": {"width": WIDTH, "height": HEIGHT,
              "responses": [{"identifier": "default", "format": {"type": "image/tiff"}}]},
  "evalscript": evalscript,
}
r = requests.post('https://sh.dataspace.copernicus.eu/api/v1/process',
                  headers={'Authorization': f'Bearer {TOKEN}'}, json=body)
if r.status_code != 200:
    raise SystemExit(f'Process API error {r.status_code}: {r.text[:500]}')
open('sar.tif', 'wb').write(r.content)
print('Downloaded real SAR scene:', len(r.content), 'bytes ✓')

In [ ]:
# 5) Detect vessels: CFAR-style bright-target detection on the radar image
import rasterio, numpy as np, math
from scipy import ndimage

with rasterio.open('sar.tif') as ds:
    vv = ds.read(1).astype('float32')
    mask = ds.read(2)
    T = ds.transform

sea = vv[(mask > 0) & (vv > 0)]
mu, sd = float(np.nanmean(sea)), float(np.nanstd(sea))
K = 6.0                        # detection sensitivity (lower = more contacts)
targets = (vv > mu + K * sd) & (mask > 0)
labels, n = ndimage.label(targets)

latc = (BBOX[1] + BBOX[3]) / 2
px_m_x = (BBOX[2] - BBOX[0]) * 111320 * math.cos(math.radians(latc)) / vv.shape[1]
px_m_y = (BBOX[3] - BBOX[1]) * 110540 / vv.shape[0]
px_m = (px_m_x + px_m_y) / 2

dets = []
for i in range(1, n + 1):
    ys, xs = np.where(labels == i)
    size = len(xs)
    if size < 2 or size > 800:  # drop speckle noise and large land/artifact blobs
        continue
    cx, cy = float(xs.mean()), float(ys.mean())
    lon, lat = T * (cx, cy)
    length = round(max(px_m, math.sqrt(size) * px_m), 1)
    conf = round(min(0.99, (float(vv[int(cy), int(cx)]) - mu) / (sd * 10) + 0.6), 2)
    dets.append({'id': f'SARR-{len(dets)}', 'lat': round(float(lat), 5),
                 'lon': round(float(lon), 5), 'length_m': length, 'confidence': conf})

print(f'{len(dets)} vessel detections (mean={mu:.4f}, std={sd:.4f}, ~{px_m:.0f} m/px)')
print('If 0 detections, lower K (e.g. 5.0) in this cell and re-run.')

In [ ]:
# 6) Print the JSON — copy ALL of this output and paste it back into the PRAHARI chat
import json
out = {'scene': 'Sentinel-1 GRD (Copernicus Data Space)', 'aoi_bbox': BBOX,
       'pixel_m': round(px_m, 1), 'count': len(dets), 'detections': dets}
print(json.dumps(out, indent=2))